# 05e · 둡 부등식과 아주마 부등식 (Doob's Maximal Inequality and the Azuma–Hoeffding Inequality)

| 항목 | 내용 |
|---|---|
| 교재 지도 | Williams §14.6 (Doob's submartingale inequality), §14.11 (Doob's $L^p$ inequality) · G&S 4e §12.2 (martingale differences, Hoeffding's inequality, bin packing), §12.6 (the maximal inequality) · Lawler 2e §5.5–5.6 · McDiarmid (1989), "On the method of bounded differences" |
| 선수 노트북 | 05d (마팅게일 수렴, UI), 05c (월드 항등식), 05a (조건부 기대·마팅게일 동물원) |
| 예상 소요 | 90분 (선택) |
| 상태 | draft |

05d까지의 마팅게일 정리는 "시각 $n$의 값 하나"에 관한 것이었다(기대값 보존, 수렴). 이 노트북의 두 부등식은 **경로 전체**에 관한 것이다. **둡 최대 부등식**(Doob's maximal inequality)은 "시각 $n$까지의 최댓값이 $\lambda$를 넘을 확률"을 **끝값 $X_n$의 1차 모멘트 하나**로 누르고, **아주마–회프딩 부등식**(Azuma–Hoeffding inequality)은 "증분이 유계인 마팅게일이 $t$만큼 벗어날 확률"을 **분포를 전혀 모른 채** 가우스형 $e^{-t^2/2\sum c_k^2}$로 누른다. 두 번째 부등식이 조합론과 알고리즘 분석에서 강력한 이유는 **둡 마팅게일**(Doob martingale) $M_k=E[f\mid\mathcal F_k]$ 때문이다: 어떤 함수 $f$든 마팅게일로 만들 수 있고, "입력 하나가 $f$를 조금밖에 못 바꾼다"는 것만 확인하면 집중 부등식이 따라온다. 우리는 단순 랜덤워크(SRW)에서 부등식과 정확값(반사 원리·이항 꼬리)을 나란히 놓고, "빈 상자 수" 예제에서 둡 마팅게일을 손으로 만들어 마팅게일 성질과 유계 차분을 숫자로 확인한 뒤, **상한이 언제 날카롭고 언제 느슨한가**를 $\sum c_k^2$ 대 실제 분산의 비로 읽는다.

## 0. 준비 (Setup)

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import Markdown
from math import comb, factorial
from fractions import Fraction
from scipy.stats import binom
from spkit import rng_for, fast, scale, setup_plots
from spkit.mc import mc_estimate, mc_proportion, compare_table, assert_close
from spkit.markov import random_walk_paths
from spkit.brownian import running_max
from spkit.plots import plot_paths

SEED, rng = rng_for("05e")
setup_plots()
N_PATHS = scale(20_000)   # FAST(SPKIT_FAST=1) 모드에서는 1/10 크기
N_STEPS = 100             # SRW 걸음 수
N_BINS, M_BALLS = 100, 100   # 빈 상자 예제: 상자 n개, 공 m개
print(f"SEED={SEED}, fast={fast()}, N_PATHS={N_PATHS}")

## 1. Recall — 지난 노트북에서

책을 덮고 먼저 답해 보세요. 답은 접힌 블록에 있습니다.

**Q1.** (05d) 마팅게일 수렴 정리의 조건과 결론은?

<details><summary>정답</summary>

$\sup_nE|M_n|<\infty$($L^1$ 유계)이면 $M_n\to M_\infty$ a.s.이고 $E|M_\infty|<\infty$. 결론은 a.s. 수렴까지이며, $E[M_\infty]=M_0$ 같은 기대값 보존은 별도로 균등적분가능(UI)이 필요하다.

</details>

**Q2.** (05d) 우도비 $L_n$이 0으로 수렴하는데 $E[L_n]=1$인 이유는?

<details><summary>정답</summary>

$L_n$이 UI가 아니기 때문이다. 질량이 확률 $\approx0$인 거대한 값($e^{n\mu^2/2}$ 규모)으로 도망가서 극한과 기대가 교환되지 않는다 — 05b의 배가 전략과 같은 현상이다.

</details>

**Q3.** (05d) $(a,b)=(2,1)$ 폴리아 항아리의 극한 분포와 $P(X_\infty\le1/2)$는?

<details><summary>정답</summary>

$X_\infty\sim\mathrm{Beta}(2,1)$, cdf $x^2$이므로 $P(X_\infty\le1/2)=1/4$.

</details>

**Q4.** (05c) $p=0.6$ 걸음이 $+5$에 도달하는 기대 시각을 월드 항등식으로 구하면?

<details><summary>정답</summary>

$E[S_T]=5=E[X_1]E[T]=0.2\,E[T]$ ⇒ $E[T]=25$. (분산은 $\sigma^2a/\mu^3=0.96\cdot5/0.008=600$.) 이 노트북의 E3에서 같은 $p=0.6$ 걸음이 "마팅게일이 아니라서" 아주마 부등식을 깨뜨리는 모습을 본다.

</details>

## 2. 이론 — 내 말로 다시 쓰기

### 2.1 정의

- **최대 과정**(maximal process): $X_n^*=\max_{k\le n}X_k$. 정리 1은 비음 서브마팅게일 $X$에 대해 $\lambda P(X_n^*\ge\lambda)\le E[X_n1_{\{X_n^*\ge\lambda\}}]\le E[X_n]$을 말한다.
- **따름정리** (마팅게일 $M$에 적용; 젠센으로 볼록 함수를 씌워 서브마팅게일을 만든다):
  (i) $P(\max_{k\le n}M_k\ge\lambda)\le E[M_n^+]/\lambda$ — $M^+=\max(M,0)$이 비음 서브마팅게일;
  (ii) **콜모고로프 부등식**(Kolmogorov's inequality): $P(\max_{k\le n}|M_k|\ge\lambda)\le E[M_n^2]/\lambda^2$ — $M^2$이 서브마팅게일;
  (iii) **둡 $L^2$ 부등식**: $E[\max_{k\le n}M_k^2]\le4E[M_n^2]$, 즉 $\|M_n^*\|_2\le2\|M_n\|_2$ (E2에서 유도).
- **아주마–회프딩**: $M$ 마팅게일, $|M_k-M_{k-1}|\le c_k$ (**결정적** 상수) ⇒ $P(M_n-M_0\ge t)\le\exp\!\big(-t^2/(2\sum_{k\le n}c_k^2)\big)$; 양쪽 꼬리는 $\times2$. SRW는 $c_k=1$이므로 $P(S_n\ge t)\le e^{-t^2/(2n)}$.
- **둡 마팅게일**: $f=f(\xi_1,\dots,\xi_m)$ 적분가능, $\mathcal F_k=\sigma(\xi_1,\dots,\xi_k)$일 때 $M_k=E[f\mid\mathcal F_k]$. 탑 성질로 마팅게일이고 $M_0=E[f]$, $M_m=f$이다 — "정보를 하나씩 공개하며 $f$의 예측을 갱신하는" 과정. **맥디아미드 부등식**(McDiarmid, 유계 차분): $\xi_i$가 **독립**이고 $f$가 $i$번째 좌표 하나를 바꿀 때 최대 $c_i$만큼만 변하면 $P(f-Ef\ge t)\le\exp(-2t^2/\sum c_i^2)$ — 아주마보다 지수가 4배 좋다.
- **빈 상자 예제**(G&S 4e §12.2 bin packing의 변형): 상자 $n$개에 공 $m$개를 독립·균등하게 던진다. $E_m$ = 시각 $m$의 빈 상자 수, $E_k^{now}$ = 공 $k$개를 던진 뒤 현재 빈 상자 수.
  $$E[E_m]=n\Big(1-\tfrac1n\Big)^m,\qquad \mathrm{Var}(E_m)=n\Big(1-\tfrac1n\Big)^m+n(n-1)\Big(1-\tfrac2n\Big)^m-n^2\Big(1-\tfrac1n\Big)^{2m}.$$
  둡 마팅게일은 명시적으로 $M_k=E[E_m\mid\mathcal F_k]=E_k^{now}(1-1/n)^{m-k}$, 차분은 $|M_k-M_{k-1}|\le c_k:=(1-1/n)^{m-k}$ (정리 3). 정확 분포는 제2종 스털링 수 $S(m,j)$로 $P(E_m=j)=\binom nj\,S(m,n-j)\,(n-j)!/n^m$.
- 이 노트북의 수치($n=m=100$): $E[E_{100}]=100\cdot0.99^{100}\approx36.60$, $\mathrm{Var}\approx9.74$ (sd $3.12$), $\sum c_k^2=\frac{1-0.99^{200}}{1-0.99^2}\approx43.52$ (조야한 $c_k=1$이면 $100$). 코드에서 다시 계산한다.

### 2.2 정리 1 — 둡 최대 부등식 (Williams §14.6, G&S 4e §12.6, Lawler 2e §5.6)

$X$가 비음 서브마팅게일이고 $\lambda>0$이면
$$\lambda\,P\Big(\max_{k\le n}X_k\ge\lambda\Big)\ \le\ E\big[X_n1_{\{\max_{k\le n}X_k\ge\lambda\}}\big]\ \le\ E[X_n].$$

가정이 왜 필요한가:
- **서브마팅게일**: 사건 $A_k$("시각 $k$에 처음 $\lambda$ 이상")가 $\mathcal F_k$에 속하므로 $E[X_k1_{A_k}]\le E[X_n1_{A_k}]$ — "한 번 넘으면 평균적으로 그 위에 머문다"가 증명의 전부다. 슈퍼마팅게일이면 부등호가 반대라 실패한다.
- **비음성**: $A_k$ 위에서 $X_k\ge\lambda$를 $X_n$으로 바꿔치기한 뒤 $\sum_kE[X_n1_{A_k}]\le E[X_n]$으로 마무리하려면 $X_n$의 음수 부분이 없어야 한다. 마팅게일 $M$에는 $|M|$, $M^+$, $M^2$(젠센으로 서브마팅게일)에 적용한다.
- **우변은 마지막 시각 $n$의 값**이다 — $\sup_k$도 $E[X_n^*]$도 아니다. 최댓값의 꼬리를 끝점의 1차 모멘트로 통제하는 것이 부등식의 내용이다.

<details><summary>증명</summary>

$A_k=\{X_k\ge\lambda,\ X_j<\lambda\ \forall j<k\}$, $k=0,1,\dots,n$이라 하자. 이들은 서로소이고 $\bigcup_kA_k=\{X_n^*\ge\lambda\}$이며, $A_k$는 시각 $k$까지의 값만으로 결정되므로 $A_k\in\mathcal F_k$이다. $A_k$ 위에서 $X_k\ge\lambda$이므로
$$\lambda P(A_k)\le E[X_k1_{A_k}]=E\big[E[X_k\mid\mathcal F_k]1_{A_k}\big]\le E\big[E[X_n\mid\mathcal F_k]1_{A_k}\big]=E[X_n1_{A_k}],$$
가운데 부등호는 서브마팅게일 성질 $X_k\le E[X_n\mid\mathcal F_k]$ (탑 성질을 $k<n$까지 반복)과 $1_{A_k}\ge0$, 마지막 등호는 $A_k\in\mathcal F_k$에서 나온다. $k$에 대해 더하면 $\lambda P(X_n^*\ge\lambda)\le E[X_n1_{\{X_n^*\ge\lambda\}}]$이고, $X_n\ge0$이므로 이는 $\le E[X_n]$이다. $\square$

따름정리: (i) $x\mapsto x^+$는 볼록·비감소이므로 조건부 젠센으로 $M^+$는 비음 서브마팅게일이고 $\{\max M_k\ge\lambda\}=\{\max M_k^+\ge\lambda\}$ ($\lambda>0$). (ii) $M\in L^2$이면 $M^2$은 비음 서브마팅게일이고 $\{\max|M_k|\ge\lambda\}=\{\max M_k^2\ge\lambda^2\}$에 정리를 쓰면 $\lambda^2P(\cdot)\le E[M_n^2]$. (iii) E2에서 layer-cake + 코시–슈바르츠로 유도.

</details>

### 2.3 정리 2 — 아주마–회프딩 부등식 (G&S 4e §12.2, McDiarmid 1989)

$M$이 마팅게일이고 $|M_k-M_{k-1}|\le c_k$ ($c_k$ 결정적 상수)이면 모든 $t>0$에 대해
$$P(M_n-M_0\ge t)\le\exp\!\Big(-\frac{t^2}{2\sum_{k\le n}c_k^2}\Big),\qquad P(|M_n-M_0|\ge t)\le2\exp\!\Big(-\frac{t^2}{2\sum_{k\le n}c_k^2}\Big).$$

가정이 왜 필요한가:
- **유계 차분**: 회프딩 보조정리 $E[e^{\theta D}\mid\mathcal F]\le e^{\theta^2c^2/2}$가 $|D|\le c$를 요구한다. 무계 증분(예: 가우스)이면 다른 보조정리가 필요하고, 꼬리가 무거우면 지수 꼬리 자체가 거짓이다.
- **마팅게일(조건부 평균 0 증분)**: 드리프트가 있으면 $E[e^{\theta D}\mid\mathcal F]$에 $e^{\theta E[D\mid\mathcal F]}$ 인자가 남아 부등식이 깨진다 (E3: $p=0.6$ 걸음, 실제 $0.543\gg e^{-2}$).
- **$c_k$는 결정적 상수**: 확률변수 $c_k$이면 $\prod_ke^{\theta^2c_k^2/2}$를 기대 밖으로 꺼낼 수 없어 $\theta$ 최적화 단계가 무너진다.

<details><summary>증명</summary>

**회프딩 보조정리.** $|d|\le c$이면 $d$는 $-c$와 $c$의 볼록 결합 $d=\frac{c-d}{2c}(-c)+\frac{c+d}{2c}(c)$이므로 $e^{\theta d}$의 볼록성에서 현(chord) 부등식
$$e^{\theta d}\le\frac{c-d}{2c}e^{-\theta c}+\frac{c+d}{2c}e^{\theta c}.$$
$d=D$를 넣고 $\mathcal F$-조건부 기대를 취하면 $E[D\mid\mathcal F]=0$이므로 $D$의 1차 항이 사라져 $E[e^{\theta D}\mid\mathcal F]\le\tfrac12(e^{-\theta c}+e^{\theta c})=\cosh(\theta c)$. 마지막으로 테일러 급수를 항별로 비교하면 $\frac{x^{2k}}{(2k)!}\le\frac{x^{2k}}{2^kk!}$ ($(2k)!\ge2^kk!$)이므로 $\cosh x\le e^{x^2/2}$. 따라서 $E[e^{\theta D}\mid\mathcal F]\le e^{\theta^2c^2/2}$.

**체르노프 논증.** $D_k=M_k-M_{k-1}$. 마르코프 부등식을 $e^{\theta(M_n-M_0)}$에 쓰면 $\theta>0$에 대해 $P(M_n-M_0\ge t)\le e^{-\theta t}E[e^{\theta(M_n-M_0)}]$. 탑 성질로 마지막 증분부터 벗겨낸다:
$$E\big[e^{\theta(M_n-M_0)}\big]=E\Big[e^{\theta(M_{n-1}-M_0)}\,E[e^{\theta D_n}\mid\mathcal F_{n-1}]\Big]\le e^{\theta^2c_n^2/2}\,E\big[e^{\theta(M_{n-1}-M_0)}\big]\le\cdots\le\exp\Big(\tfrac{\theta^2}{2}\sum_kc_k^2\Big).$$
여기서 $e^{\theta(M_{n-1}-M_0)}$이 $\mathcal F_{n-1}$-가측이고 $c_k$가 상수라서 기대 밖으로 나온다. 그러므로 $P(M_n-M_0\ge t)\le\exp(\tfrac{\theta^2}{2}\sum c_k^2-\theta t)$이고, 지수는 $\theta=t/\sum c_k^2$에서 최소 $-t^2/(2\sum c_k^2)$이다. 아래 꼬리는 $-M$에 적용하고, 양쪽 꼬리는 합집합 경계로 $\times2$. $\square$

SRW 수치($n=100$, $t=20$): 상한 $e^{-400/200}=e^{-2}\approx0.1353$, 정확값 $P(\mathrm{Bin}(100,\tfrac12)\ge60)\approx0.0284$ — 상한이 5배 크다. 지수 $t^2/2n$은 맞지만 다항 인자($\sim1/(t\sqrt{2\pi/n})$ 규모)를 잃는다.

</details>

### 2.4 정리 3 — 빈 상자의 둡 마팅게일 (G&S 4e §12.2의 bin packing 논법)

상자 $n$개, 공 $m$개일 때 $M_k=E[E_m\mid\mathcal F_k]=E_k^{now}(1-1/n)^{m-k}$이고 $|M_k-M_{k-1}|\le(1-1/n)^{m-k}=:c_k$. 따라서
$$P\big(E_m-E[E_m]\ge t\big)\le\exp\Big(-\frac{t^2}{2\sum_{k=1}^mc_k^2}\Big),\qquad \sum_{k=1}^mc_k^2=\sum_{j=0}^{m-1}\Big(1-\tfrac1n\Big)^{2j}=\frac{1-(1-1/n)^{2m}}{1-(1-1/n)^2}.$$

가정이 왜 필요한가:
- **공 던지기의 독립·균등성**: "상자 하나가 남은 $m-k$개 공을 모두 피할 확률 $(1-1/n)^{m-k}$"가 여기서 나온다. 공끼리 상관이 있으면 $M_k$의 닫힌 형식도, $c_k$도 달라진다.
- **차분 유계**: 공 하나는 현재 빈 상자 수를 최대 1 줄이므로 $M$의 변화는 $\le(1-1/n)^{m-k}$. 조야하게 $c_k=1$로 두어도 부등식은 성립하지만 $\sum c_k^2=m$으로 상한이 더 약해진다.

<details><summary>증명</summary>

**마팅게일의 닫힌 형식.** $E_m=\sum_{i=1}^n1_{\{i\text{가 시각 }m\text{에 빔}\}}$이므로 $E[E_m\mid\mathcal F_k]=\sum_iP(i\text{가 시각 }m\text{에 빔}\mid\mathcal F_k)$. 시각 $k$에 이미 찬 상자는 그 확률이 0이고, 현재 빈 상자는 남은 $m-k$개 공이 독립·균등하게 모두 피해야 하므로 $(1-1/n)^{m-k}$. 현재 빈 상자가 $E_k^{now}$개이므로 $M_k=E_k^{now}(1-1/n)^{m-k}$. ($M_0=n(1-1/n)^m=E[E_m]$, $M_m=E_m$을 확인하라.)

**차분 유계.** $M_k-M_{k-1}=(1-\tfrac1n)^{m-k}\big[E_k^{now}-(1-\tfrac1n)E_{k-1}^{now}\big]=(1-\tfrac1n)^{m-k}\big[(E_k^{now}-E_{k-1}^{now})+E_{k-1}^{now}/n\big]$. $k$번째 공은 빈 상자에 떨어지거나($E_k^{now}=E_{k-1}^{now}-1$) 찬 상자에 떨어지므로($E_k^{now}=E_{k-1}^{now}$) 대괄호는 $\{-1+E_{k-1}^{now}/n,\ E_{k-1}^{now}/n\}\subset[-1,1]$ ($0\le E_{k-1}^{now}\le n$). 따라서 $|M_k-M_{k-1}|\le(1-1/n)^{m-k}$. 정리 2를 적용하고 $M_0=E[E_m]$, $M_m=E_m$을 넣으면 꼬리 부등식이다. $\square$

수치($n=m=100$): $\sum c_k^2\approx43.52$인데 실제 $\mathrm{Var}(E_{100})\approx9.74$이므로 아주마의 "분산 대용"이 약 4.5배 과대 — 상한이 느슨한 이유다. 반면 SRW는 $\sum c_k^2=n=\mathrm{Var}(S_n)$으로 지수가 최적이다.

</details>

### 2.5 직관

**둡 부등식**: 마팅게일은 한 번 $\lambda$를 넘으면 그 뒤 평균이 $\lambda$ 이상에 머무르므로 "최댓값이 $\lambda$를 넘을 확률"은 "끝값의 크기"로 통제된다 — 경로 전체의 최댓값의 꼬리를 끝점 하나로 잡는 놀라운 절약이다. 06c에서는 같은 원리가 브라운 운동의 반사 원리·최댓값 분포로 이어진다.

**아주마**: 유계 증분의 마팅게일은 $n$스텝 뒤 $\sqrt{\sum c_k^2}$ 규모로만 벗어난다. 분포를 몰라도, 증분 사이의 상관 구조를 몰라도, "한 걸음의 영향이 작다"는 것만으로 가우스형 집중을 얻는다. 둡 마팅게일과 결합하면 "입력 하나가 $f$를 조금밖에 못 바꾼다"만 확인하면 어떤 복잡한 $f$(빈 상자 수, 최장 증가 부분열, 그래프의 채색수 …)에도 집중 부등식이 따라온다 — 이것이 조합론·알고리즘 분석에서 둡 마팅게일이 강력한 이유다. **대가**: $\sum c_k^2$가 실제 분산보다 크면 지수가 손해 본다. SRW는 $\sum c_k^2=n=\mathrm{Var}(S_n)$이라 지수가 최적이고(정확값과의 차이는 다항 인자뿐), 빈 상자는 4.5배 손해다.

✍️ 내 말로: "왜 아주마 부등식은 SRW에서는 거의 정확하고 빈 상자에서는 느슨한가"를 $\sum c_k^2$와 분산의 관계로 두 문장으로 써 보세요.

### 2.6 함정

- **아주마는 마팅게일(평균 0 증분) 전용이다.** 드리프트가 있는 $p=0.6$ 걸음에 그대로 쓰면 $P(S_{100}\ge20)\approx0.543\gg e^{-2}$로 거짓이다. 중심화 $S_n-0.2n$과 $c=1.2$로 다시 써야 한다(E3).
- **부등식은 상한이지 근사가 아니다.** SRW $n=100$, $t=20$: 상한 $0.135$ vs 실제 $0.028$(5배). 빈 상자 $t=8$: 상한 $0.479$ vs 실제 $0.0056$(85배).
- **둡 부등식 우변은 마지막 시각의 $E[X_n]$**이지 $\sup_k$나 $E[X_n^*]$가 아니다 — 그것이 부등식의 내용이다.
- **$P(\max_{k\le n}S_k\ge a)$는 반사 원리로 정확히** $2P(S_n\ge a+1)+P(S_n=a)=P(S_n\ge a)+P(S_n\ge a+1)$**이다.** 부등식과 정확값을 혼동하지 말 것(06c에서 연속판).
- **맥디아미드 상수 $2t^2/\sum c^2$는 좌표 독립성을 추가로 쓴 결과**이고, 일반 마팅게일에는 아주마의 $t^2/(2\sum c^2)$만 보장된다.
- **$\sum c_k^2$의 $c_k$는 결정적 상수.** 경로마다 다른 "실제 증분 크기"를 넣으면 안 된다.

✍️ 내 말로: (책을 덮고 여기에 핵심 정의·정리를 다시 써 보세요)

## 3. Predict — 코드를 돌리기 전에 예측

코드를 돌리기 전에 아래 값을 먼저 적어 보세요. 정확할 필요는 없습니다 — "얼마나 틀렸는가"가 이 노트북의 학습 신호입니다. 모두 $n=100$ 걸음의 SRW, 또는 상자 100개·공 100개의 빈 상자 예제입니다.

In [ ]:
predictions = {
    # P(max_{k<=100} S_k >= 15)는? (n=100 SRW의 누적 최댓값; 둡·콜모고로프 상한과의 비교는 5절 표에서)
    "p_max_ge15": None,
    # E[max_{k<=100} S_k^2]는? (둡 L^2 상한 4·E[S_100^2] = 400 과 비교)
    "e_max_sq": None,
    # P(S_100 >= 20)의 정확값은? (아주마 상한은 e^{-20^2/200} = e^{-2} ≈ 0.135)
    "azuma_srw": None,
    # 상자 100개에 공 100개를 던졌을 때 빈 상자 수의 기대값은?
    "empty_mean": None,
    # P(E_100 - E[E_100] >= 8)의 정확값은? (날카로운 c_k 아주마 상한은 약 0.48)
    "empty_tail8": None,
}

## 4. Simulate — 시뮬레이션

### 4.0 helper — 정확값과 상한 (spkit에 없어서 여기서 정의)

SRW의 최댓값 분포는 두 가지로 정확히 계산한다. $\max S_k$는 **반사 원리**, $\max|S_k|$는 $(-a,a)$ 밖으로 나가면 죽는 **killed walk**의 생존 질량을 동적계획법(DP)으로 구해 $1-$생존. $E[\max S_k^2]=\sum_{a\ge1}(2a-1)P(\max|S_k|\ge a)$는 비음 정수 확률변수 $Y=\max|S_k|$에 대한 $E[Y^2]=\sum_a(2a-1)P(Y\ge a)$이다.

In [ ]:
# helper (spkit에 없어서 여기서 정의)
def srw_tail(a, n):
    """P(S_n >= a), S_n = 2K - n, K ~ Bin(n, 1/2). a는 스칼라 또는 배열."""
    a = np.asarray(a, dtype=float)
    k_min = np.ceil((n + a) / 2)             # S_n >= a  <=>  K >= (n+a)/2
    return np.where(a <= -n, 1.0, binom.sf(k_min - 1, n, 0.5))

def reflection_tail(a, n):
    """P(max_{k<=n} S_k >= a) = P(S_n >= a) + P(S_n >= a+1)  (반사 원리; a >= 1)."""
    return srw_tail(a, n) + srw_tail(np.asarray(a) + 1, n)

def absmax_tail_dp(a, n):
    """P(max_{k<=n} |S_k| >= a): 상태 -a+1..a-1 위의 killed walk 생존 질량으로 계산."""
    a = int(a)
    if a <= 0:
        return 1.0
    v = np.zeros(2 * a - 1); v[a - 1] = 1.0
    for _ in range(n):
        nv = np.zeros_like(v)
        nv[1:] += 0.5 * v[:-1]; nv[:-1] += 0.5 * v[1:]
        v = nv
    return 1.0 - v.sum()

def azuma_bound(t, c2_sum):      return np.exp(-np.asarray(t, float) ** 2 / (2 * c2_sum))
def mcdiarmid_bound(t, c2_sum):  return np.exp(-2 * np.asarray(t, float) ** 2 / c2_sum)

빈 상자 예제의 helper: `count_empty_bins`는 시각 $m$의 빈 상자 수, `doob_empty_martingale`은 공을 하나씩 공개하며 $E_k^{now}$와 $M_k=E_k^{now}(1-1/n)^{m-k}$를 계산한다. 정확 분포는 제2종 스털링 수 $S(m,j)$의 점화식 $S(i,k)=kS(i-1,k)+S(i-1,k-1)$을 정수 산술로 돌려 유리수로 구한다(부동소수점 오차 없음).

In [ ]:
# helper (spkit에 없어서 여기서 정의)
def count_empty_bins(balls, n):
    """balls: (reps, m) 정수 배열 → 각 복제의 빈 상자 수 E_m."""
    occupied = np.zeros((balls.shape[0], n), bool)
    np.put_along_axis(occupied, balls, True, axis=1)
    return n - occupied.sum(1)

def doob_empty_martingale(balls, n):
    """(M, E_now): 둡 마팅게일 M_k = E_k^now (1-1/n)^(m-k) 와 현재 빈 상자 수, k = 0..m."""
    reps, m = balls.shape
    seen = np.zeros((reps, n), bool)
    E_now = np.empty((reps, m + 1), int); E_now[:, 0] = n
    for k in range(m):
        seen[np.arange(reps), balls[:, k]] = True
        E_now[:, k + 1] = n - seen.sum(1)
    weights = (1 - 1 / n) ** (m - np.arange(m + 1))
    return E_now * weights, E_now

def empty_bins_pmf_exact(n, m):
    """dict j -> P(E_m = j) (Fraction): P = C(n,j) S(m,n-j) (n-j)! / n^m, S = 제2종 스털링 수."""
    S = [[0] * (n + 1) for _ in range(m + 1)]; S[0][0] = 1
    for i in range(1, m + 1):
        for k in range(1, n + 1):
            S[i][k] = k * S[i - 1][k] + S[i - 1][k - 1]
    return {j: Fraction(comb(n, j) * S[m][n - j] * factorial(n - j), n ** m) for j in range(n + 1)}

### 4.1 SRW의 최댓값 — 둡·콜모고로프 상한 vs 정확값 (Fig 1, 2)

먼저 경로와 그 **누적 최댓값**(running maximum)을 그려 둡 부등식이 무엇을 통제하는지 눈으로 본다. 둡 부등식은 검은 계단($\max_{k\le n}S_k$)의 끝값이 $\lambda$를 넘을 확률을, 얇은 경로($S_n$)의 끝값의 양수 부분 평균으로 누른다.

In [ ]:
paths = random_walk_paths(N_PATHS, N_STEPS, rng)          # (N_PATHS, 101), S_0 = 0
runmax = running_max(paths)
maxS, maxAbsS, S_n = runmax[:, -1], running_max(np.abs(paths))[:, -1], paths[:, -1]
t_grid = np.arange(N_STEPS + 1)

fig, ax = plt.subplots()
plot_paths(t_grid, paths, ax=ax, n_show=3, alpha=0.6)
for i in range(3):
    ax.step(t_grid, runmax[i], where="post", color="k", lw=1.2, alpha=0.9)
ax.axhline(15, color="C3", ls="--", label="lambda = 15")
ax.set(xlabel="k", ylabel="S_k  and  max_{j<=k} S_j (black steps)",
       title="Fig 1. SRW paths (thin) and their running maxima (black)")
ax.legend(loc="lower left"); plt.show()

이제 $\lambda=2,4,\dots,30$에 대해 두 가지 최댓값의 꼬리를 MC·정확값·상한으로 비교한다. 왼쪽은 $\max S_k$(둡 (i): $E[S_n^+]/\lambda$), 오른쪽은 $\max|S_k|$(콜모고로프: $E[S_n^2]/\lambda^2=n/\lambda^2$). $E[S_n^+]$는 이항 분포에서 정확히 계산한다.

In [ ]:
ks = np.arange(N_STEPS + 1); S_vals = 2 * ks - N_STEPS; pm = binom.pmf(ks, N_STEPS, 0.5)
E_Splus = float((np.maximum(S_vals, 0) * pm).sum())            # E[S_n^+]
lams = np.arange(2, 31, 2)
mc_max = np.array([mc_proportion(maxS >= l).mean for l in lams])
mc_absmax = np.array([mc_proportion(maxAbsS >= l).mean for l in lams])
exact_max = reflection_tail(lams, N_STEPS)
exact_absmax = np.array([absmax_tail_dp(l, N_STEPS) for l in lams])

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for ax, mc, ex, bound, ttl, bl in [
    (axes[0], mc_max, exact_max, E_Splus / lams, "P(max_{k<=100} S_k >= lambda)", "Doob bound E[S_n^+]/lambda"),
    (axes[1], mc_absmax, exact_absmax, N_STEPS / lams**2, "P(max_{k<=100} |S_k| >= lambda)", "Kolmogorov bound n/lambda^2"),
]:
    ax.semilogy(lams, mc, "o", label="MC"); ax.semilogy(lams, ex, "-", label="exact")
    ax.semilogy(lams, np.minimum(bound, 1), "--", label=bl)
    ax.set(title=ttl, xlabel="lambda", ylim=(1e-4, 1.5)); ax.legend()
fig.suptitle("Fig 2. Maximal inequalities vs exact tails (SRW, n = 100)"); plt.tight_layout(); plt.show()
print(f"E[S_100^+] = {E_Splus:.4f};  lambda=15: exact {float(reflection_tail(15, N_STEPS)):.4f} <= Doob {E_Splus/15:.4f};"
      f"  |S|: exact {absmax_tail_dp(15, N_STEPS):.4f} <= Kolmogorov {N_STEPS/15**2:.4f}")

상한(점선)은 항상 정확값(실선) 위에 있지만 $\lambda$가 작을 때는 1을 넘어 무의미하고, 큰 $\lambda$에서는 정확값이 가우스 꼬리로 떨어지는 동안 상한은 $1/\lambda$, $1/\lambda^2$로만 떨어진다 — 모멘트 하나로 얻는 상한의 한계다.

### 4.2 둡 $L^2$ 부등식 — $E[\max S_k^2]\le4E[S_n^2]=400$

$L^2$ 부등식은 꼬리가 아니라 **최댓값의 2차 모멘트**를 끝값의 2차 모멘트로 누른다. 정확값은 4.0의 DP를 $a=1,\dots,n$에 대해 합해 얻는다. 덤으로 $E[\max_{k\le n}S_k]=\sum_{a\ge1}P(\max S_k\ge a)$(반사 원리의 합)도 같이 본다.

In [ ]:
a_all = np.arange(1, N_STEPS + 1)
absmax_tails = np.array([absmax_tail_dp(a, N_STEPS) for a in a_all])
E_maxsq_exact = float(((2 * a_all - 1) * absmax_tails).sum())
E_absmax_exact = float(absmax_tails.sum())
E_max_exact = float(reflection_tail(a_all, N_STEPS).sum())
doob_L2 = 4 * N_STEPS                                              # 4 E[S_n^2] = 4n

rows_L2 = [
    {"name": "E[max S_k^2]  (Doob L2 bound 400)", "predicted": predictions["e_max_sq"],
     "estimate": mc_estimate(maxAbsS**2), "exact": E_maxsq_exact},
    {"name": "E[max ∣S_k∣]  (bound 2 sqrt(n) = 20)", "predicted": None,
     "estimate": mc_estimate(maxAbsS), "exact": E_absmax_exact},
    {"name": "E[max S_k]", "predicted": None, "estimate": mc_estimate(maxS), "exact": E_max_exact},
]
Markdown(compare_table(rows_L2))

$E[\max S_k^2]\approx171$은 상한 400의 43% — $L^2$ 부등식의 상수 4는 최악의 경우를 위한 것이고 SRW에서는 2.3배 여유가 있다. 상한이 무의미해 보여도 "어떤 마팅게일이든 $n$에 무관한 상수 4"가 핵심이다.

### 4.3 아주마 on SRW — 이항 꼬리 vs $e^{-t^2/2n}$ (Fig 3)

SRW는 $c_k=1$이라 $\sum c_k^2=n=\mathrm{Var}(S_n)$, 즉 아주마의 "분산 대용"이 실제 분산과 정확히 같다. 상한이 어디까지 정확값을 따라가는지 $t=0,\dots,40$에서 본다.

In [ ]:
t_srw = np.arange(0, 41)
exact_srw = srw_tail(t_srw, N_STEPS)
azuma_srw = azuma_bound(t_srw, N_STEPS)
mc_srw = [mc_proportion(S_n >= t) for t in t_srw]

fig, ax = plt.subplots()
ax.semilogy(t_srw, exact_srw, "-", label="exact binomial P(S_100 >= t)")
ax.errorbar(t_srw, [r.mean for r in mc_srw], yerr=[1.96 * r.se for r in mc_srw], fmt="o", ms=3, label="MC (95% CI)")
ax.semilogy(t_srw, azuma_srw, "--", label="Azuma bound exp(-t^2/200)")
ax.set(xlabel="t", ylim=(1e-5, 1.5), title="Fig 3. Tail P(S_100 >= t): exact, MC, Azuma bound"); ax.legend(); plt.show()
for t in (10, 20, 30):
    print(f"t={t:2d}: MC {mc_srw[t]}  exact {exact_srw[t]:.6f}  Azuma {azuma_srw[t]:.6f}  ratio {azuma_srw[t]/exact_srw[t]:.1f}")

로그 축에서 정확값과 상한이 **같은 기울기(같은 포물선)**로 떨어진다 — 지수 $t^2/2n$이 최적이라는 뜻이다. 둘 사이의 간격(비 3~6배, $t=10..30$)은 $t\sqrt{2\pi/n}$ 규모로 $t$에 비례해 자라는 다항 인자다 — 잃는 것은 지수가 아니라 다항 인자뿐이다.

### 4.4 빈 상자 — 둡 마팅게일을 손으로 만들기 (Fig 4, 5)

상자 100개에 공 100개. 공을 하나씩 "공개"하며 $M_k=E_k^{now}\cdot0.99^{100-k}$를 계산한다. $M_0=100\cdot0.99^{100}\approx36.60$(상수)에서 출발해 $M_{100}=E_{100}$(실제 빈 상자 수)에 도착하는 경로들을 본다. 복제 수는 SRW의 2배($2N_{PATHS}$)다 — 5절에서 $\mathrm{Var}(E_{100})$을 MC로 추정하는데, 분산 추정량의 SE는 4차 모멘트에 의존해 평균·비율의 SE보다 무겁기 때문이다.

In [ ]:
balls = rng.integers(0, N_BINS, size=(2 * N_PATHS, M_BALLS))   # 복제 2배: 5절의 Var(E_100) MC 추정을 위해
M, E_now = doob_empty_martingale(balls, N_BINS)
E100 = count_empty_bins(balls, N_BINS)
assert np.array_equal(E100, E_now[:, -1]) and np.allclose(M[:, -1], E100)   # M_m = f
mu_E = N_BINS * (1 - 1 / N_BINS) ** M_BALLS                                 # E[E_m] = M_0
c_k = (1 - 1 / N_BINS) ** (M_BALLS - np.arange(1, M_BALLS + 1))            # 차분 상한 c_k
max_diff = np.abs(np.diff(M, axis=1)).max(axis=0)                           # 경로 전체의 실제 최대 |M_k - M_{k-1}|
assert np.all(max_diff <= c_k + 1e-12), "bounded-difference violated"

fig, ax = plt.subplots()
plot_paths(np.arange(M_BALLS + 1), M, ax=ax, n_show=10, alpha=0.7)
ax.axhline(mu_E, color="k", ls="--", label=f"M_0 = E[E_100] = {mu_E:.2f}")
ax.set(xlabel="k (balls revealed)", ylabel="M_k", title="Fig 4. Doob martingale M_k = E_k^now (1-1/n)^(m-k), 10 replicates")
ax.legend(); plt.show()
print(f"E[E_100] MC {mc_estimate(E100)}  vs  M_0 = {mu_E:.4f};  max_k |dM_k| / c_k over all paths = {(max_diff / c_k).max():.4f}")

마팅게일 성질 $E[M_{k+1}\mid\mathcal F_k]=M_k$를 **조건부로** 검증한다: $k=50$에서 $E_{50}^{now}$ 값별로 경로를 묶고(bin), 각 묶음 안에서 $M_{51}$의 평균이 $M_{50}=e\cdot0.99^{50}$과 일치하는지 본다. 가장 많은 경로가 모인 값 $e^*$에서 SE가 가장 작다.

In [ ]:
K = 50
E50, M51 = E_now[:, K], M[:, K + 1]
vals, counts = np.unique(E50, return_counts=True)
e_star = int(vals[counts.argmax()])
print(f"E_50^now modal value e* = {e_star} ({counts.max()} paths);  M_50 = e*·0.99^50 = {e_star * 0.99**K:.4f}")
print(" e   #paths   mean M_51 (± se)    M_50 = e·0.99^50")
for e in vals[np.argsort(-counts)[:5]]:
    r = mc_estimate(M51[E50 == e])
    print(f"{e:3d}  {r.n:6d}   {r.mean:8.4f} ± {r.se:.4f}   {e * 0.99**K:8.4f}")
cond_check = mc_estimate(M51[E50 == e_star])

이제 꼬리 $P(E_{100}-\mu\ge t)$, $t=0,\dots,15$를 네 가지 상한과 비교한다: 정확값(스털링 수), 아주마-날카로움($c_k=0.99^{100-k}$), 아주마-조야($c_k=1$), 맥디아미드(좌표 독립·$c_i=1$: $\exp(-2t^2/100)$).

In [ ]:
pmf_exact = empty_bins_pmf_exact(N_BINS, M_BALLS)                  # dict j -> Fraction
assert abs(float(sum(pmf_exact.values())) - 1) < 1e-12
mean_exact = float(sum(j * p for j, p in pmf_exact.items()))
var_exact = float(sum(j * j * p for j, p in pmf_exact.items())) - mean_exact**2
assert abs(mean_exact - mu_E) < 1e-9
var_formula = mu_E + N_BINS * (N_BINS - 1) * (1 - 2 / N_BINS) ** M_BALLS - N_BINS**2 * (1 - 1 / N_BINS) ** (2 * M_BALLS)
c2_sharp = float((c_k**2).sum()); c2_closed = (1 - (1 - 1/N_BINS) ** (2*M_BALLS)) / (1 - (1 - 1/N_BINS) ** 2)
assert abs(c2_sharp - c2_closed) < 1e-9
print(f"exact mean {mean_exact:.6f} (formula {mu_E:.6f}), exact var {var_exact:.6f} (formula {var_formula:.6f}), sum c_k^2 = {c2_sharp:.4f}")

t_bins = np.arange(0, 16)
exact_tail = np.array([float(sum(p for j, p in pmf_exact.items() if j - mu_E >= t)) for t in t_bins])
mc_tail = [mc_proportion(E100 - mu_E >= t) for t in t_bins]
bounds = {"Azuma sharp c_k": azuma_bound(t_bins, c2_sharp), "Azuma c_k = 1": azuma_bound(t_bins, M_BALLS),
          "McDiarmid (indep., c_i = 1)": mcdiarmid_bound(t_bins, M_BALLS)}
fig, ax = plt.subplots()
ax.semilogy(t_bins, exact_tail, "-", label="exact (Stirling numbers)")
ax.errorbar(t_bins, [r.mean for r in mc_tail], yerr=[1.96 * r.se for r in mc_tail], fmt="o", ms=3, label="MC (95% CI)")
for (lbl, b), ls in zip(bounds.items(), ["--", ":", "-."]):
    ax.semilogy(t_bins, b, ls, label=lbl)
ax.set(xlabel="t", ylim=(1e-6, 1.5), title="Fig 5. Upper tail P(E_100 - 36.60 >= t): MC, exact, bounds"); ax.legend(); plt.show()

세 상한 모두 정확값 위에 있지만 간격이 크다: $t=8$에서 정확값 $\approx0.0056$ vs 날카로운 아주마 $\approx0.48$(85배). 맥디아미드가 조야한 아주마보다 4배 좋은 지수를 갖지만, 그래도 실제 꼬리에는 한참 못 미친다 — 문제는 $c_k$의 합이 아니라 **$\sum c_k^2$가 실제 분산을 4.5배 과대평가한다**는 데 있다.

### 4.5 부등식 성립 확인과 "지수 비교"

부등식은 확률적 진술이 아니라 **항상** 성립해야 하므로, 모든 $t$ 격자에서 MC 꼬리의 신뢰구간 하한($\hat p-4\,\mathrm{se}$)이 각 상한 아래에 있는지 약하게 검사한다. 그리고 두 예제에서 $\sum c_k^2/\mathrm{Var}$를 표로 놓는다 — 이 비가 1이면 지수가 최적, 클수록 상한이 느슨하다.

In [ ]:
for name, b in bounds.items():
    assert all(r.mean - 4 * r.se <= bi for r, bi in zip(mc_tail, b)), f"{name} violated"
assert all(r.mean - 4 * r.se <= bi for r, bi in zip(mc_srw, azuma_srw)), "Azuma (SRW) violated"
r_maxsq = mc_estimate(maxAbsS**2)
assert r_maxsq.mean - 4 * r_maxsq.se <= doob_L2, "Doob L2 violated"
print("all bounds hold on the grid (MC lower CI <= bound)")

ratio_rows = [("SRW S_100 (c_k = 1)", N_STEPS, N_STEPS),
              ("empty bins, sharp c_k = 0.99^(100-k)", c2_sharp, var_exact),
              ("empty bins, crude c_k = 1", float(M_BALLS), var_exact)]
lines = ["| 마팅게일 | Σ c_k² | Var(M_n) | Σ c_k² / Var | Azuma 상한 / 정확값 (t = 2 sd) |", "|---|---|---|---|---|"]
for name, c2, var in ratio_rows:
    t2 = 2 * np.sqrt(var)
    ex = float(srw_tail(t2, N_STEPS)) if name.startswith("SRW") else float(sum(p for j, p in pmf_exact.items() if j - mu_E >= t2))
    lines.append(f"| {name} | {c2:.2f} | {var:.2f} | {c2/var:.2f} | {azuma_bound(t2, c2)/ex:.1f}× |")
Markdown("\n".join(lines))

## 5. Compare — 예측 · 시뮬레이션 · 닫힌 형식 비교

닫힌 형식은 모두 위에서 코드로 계산한 값이다(반사 원리, killed-walk DP, 이항 꼬리, 빈 상자의 닫힌 식과 스털링 수 분포). 내 예측은 3절의 `predictions`에서 가져온다.

In [ ]:
p_max15_exact = float(reflection_tail(15, N_STEPS))
p_absmax15_exact = absmax_tail_dp(15, N_STEPS)
p_S20_exact = float(srw_tail(20, N_STEPS))
tail4_exact = float(sum(p for j, p in pmf_exact.items() if j - mu_E >= 4))
tail8_exact = float(sum(p for j, p in pmf_exact.items() if j - mu_E >= 8))
r_max15, r_absmax15 = mc_proportion(maxS >= 15), mc_proportion(maxAbsS >= 15)
r_maxS, r_S20 = mc_estimate(maxS), mc_proportion(S_n >= 20)
r_mean, r_var = mc_estimate(E100), mc_estimate((E100 - E100.mean()) ** 2)
r_tail4, r_tail8 = mc_proportion(E100 - mu_E >= 4), mc_proportion(E100 - mu_E >= 8)
P = predictions
rows = [
    {"name": f"P(max S_k >= 15)  (Doob {E_Splus/15:.3f})", "predicted": P["p_max_ge15"], "estimate": r_max15, "exact": p_max15_exact},
    {"name": f"P(max ∣S_k∣ >= 15)  (Kolmogorov {N_STEPS/15**2:.3f})", "predicted": None, "estimate": r_absmax15, "exact": p_absmax15_exact},
    {"name": "E[max S_k^2]  (Doob L2 bound 400)", "predicted": P["e_max_sq"], "estimate": r_maxsq, "exact": E_maxsq_exact},
    {"name": "E[max S_k]", "predicted": None, "estimate": r_maxS, "exact": E_max_exact},
    {"name": f"P(S_100 >= 20)  (Azuma {azuma_srw[20]:.4f})", "predicted": P["azuma_srw"], "estimate": r_S20, "exact": p_S20_exact},
    {"name": "E[E_100] empty bins", "predicted": P["empty_mean"], "estimate": r_mean, "exact": mu_E},
    {"name": "Var(E_100)", "predicted": None, "estimate": r_var, "exact": var_exact},
    {"name": f"P(E_100 - mu >= 4)  (Azuma sharp {azuma_bound(4, c2_sharp):.3f})", "predicted": None, "estimate": r_tail4, "exact": tail4_exact},
    {"name": f"P(E_100 - mu >= 8)  (Azuma sharp {azuma_bound(8, c2_sharp):.3f})", "predicted": P["empty_tail8"], "estimate": r_tail8, "exact": tail8_exact},
    {"name": f"E[M_51 ∣ E_50^now = {e_star}]  (= M_50)", "predicted": None, "estimate": cond_check, "exact": e_star * 0.99**K},
]
Markdown(compare_table(rows))

각 행에서 불일치가 뜻하는 것:
- **P(max S_k ≥ 15)**: 반사 원리 공식(짝홀 처리) 또는 `running_max`의 축 방향이 틀렸다.
- **P(max |S_k| ≥ 15)**: killed-walk DP의 경계(상태 $-a+1..a-1$) 처리가 틀렸다.
- **E[max S_k²]**: $E[Y^2]=\sum_a(2a-1)P(Y\ge a)$ 항등식 또는 DP 합의 범위가 틀렸다.
- **E[max S_k]**: 반사 원리 꼬리의 합 $\sum_aP(\max S\ge a)$가 틀렸다.
- **P(S_100 ≥ 20)**: $S_n=2K-n$의 $K$ 변환(올림) 오류.
- **E[E_100]**: 공 던지기가 균등하지 않거나 `put_along_axis` 계산이 잘못됐다.
- **Var(E_100)**: 분산 공식의 두 번째 항($(1-2/n)^m$: 두 상자가 모두 비는 확률)이 틀렸다. 단, 분산 추정량의 SE는 4차 모멘트에 의존해 평균보다 무거워 고정 시드에서 2~3 SE 벗어나는 일이 드물지 않다(그래서 복제를 2배로 했다) — 시드를 바꿔 여러 번 돌렸을 때 편향이 없으면 공식은 맞는 것이다.
- **P(E_100 − μ ≥ 4), ≥ 8**: 스털링 수 점화식 또는 $j\leftrightarrow n-j$ 대응이 틀렸다(작은 확률일수록 MC의 상대 오차 $\mathrm{SE}/p$가 커진다).
- **E[M_51 | E_50 = e*]**: 둡 마팅게일의 닫힌 형식 $E_k^{now}(1-1/n)^{m-k}$이 마팅게일이 아니라는 뜻 — 정리 3 증명의 $(1-1/n)$ 인자를 다시 보라.

In [ ]:
assert_close(r_max15, p_max15_exact, k=4, name="P(max S >= 15)")
assert_close(r_absmax15, p_absmax15_exact, k=4, name="P(max |S| >= 15)")
assert_close(r_maxsq, E_maxsq_exact, k=4, name="E[max S_k^2]")
assert_close(r_maxS, E_max_exact, k=4, name="E[max S_k]")
assert_close(r_S20, p_S20_exact, k=4, name="P(S_100 >= 20)")
assert_close(r_mean, mu_E, k=4, name="E[empty bins]")
assert_close(r_var, var_exact, k=4, name="Var(empty bins)")
assert_close(r_tail4, tail4_exact, k=4, name="P(E - mu >= 4)")
assert_close(r_tail8, tail8_exact, k=4, name="P(E - mu >= 8)")
assert_close(cond_check, e_star * 0.99**K, k=4, name="Doob martingale conditional check")
print("all assert_close passed (k=4)")

## 6. 연습문제

### E1. 계산 — $n=400$, $t=40$

(a) 아주마 상한과 정확한 $P(S_{400}\ge40)$을 구하라. (b) 둡 상한 $E[S_{400}^+]/40$과 정확한 $P(\max_{k\le400}S_k\ge40)$을 구하라. (c) $n$을 4배, $t$를 2배로 늘렸을 때 "상한/정확값" 비가 어떻게 변하는지 $n=100$, $t=20$과 비교하라.

In [ ]:
# 여기에 풀이

<details><summary>정답</summary>

(a) $\sum c_k^2=400$이므로 아주마 상한은 $e^{-1600/800}=e^{-2}\approx0.135335$ — $n=100,t=20$과 **같다**($t^2/2n$이 고정). 정확값은 $P(\mathrm{Bin}(400,\tfrac12)\ge220)\approx0.025520$.

(b) $E|S_{400}|\approx15.9477$이므로 $E[S_{400}^+]=E|S_{400}|/2\approx7.97386$, 둡 상한 $7.97386/40\approx0.199347$. 정확값은 반사 원리로 $P(S_{400}\ge40)+P(S_{400}\ge41)=2P(S_{400}\ge41)+P(S_{400}=40)\approx0.045635$.

(c) 같은 $t$끼리 비교해야 한다(4.1의 $\lambda=15$와 섞으면 안 된다). $n=100$, $t=20$의 둡 상한은 $E[S_{100}^+]/20=3.9795/20\approx0.1990$, 정확값은 $P(S_{100}\ge20)+P(S_{100}\ge21)\approx0.0460$. 아주마 비: $0.1353/0.0284\approx4.8$ → $0.1353/0.0255\approx5.3$. 둡 비: $0.1990/0.0460\approx4.3$ → $0.1993/0.0456\approx4.4$. $n$을 4배, $t$를 2배로 늘리면 $x=t/\sqrt n=2$가 고정되므로 (CLT의 연속 극한에서) 상한도 정확값도 상수로 수렴하고, 비도 상수로 수렴한다: 둡 비 → $\dfrac{1/(x\sqrt{2\pi})}{2\bar\Phi(x)}\approx4.38$, 아주마 비 → $\dfrac{e^{-x^2/2}}{\bar\Phi(x)}\approx5.95$ ($\bar\Phi$ = 표준정규 꼬리). 잃는 것은 $n$의 거듭제곱이 아니라 $x$에 대한 다항 인자($\bar\Phi(x)\approx e^{-x^2/2}/(x\sqrt{2\pi})$의 $x\sqrt{2\pi}$)다 — 부등식의 지수는 옳고, 손해는 $n$이 커져도 커지지 않는다.

```python
from scipy.stats import norm
def ratios(n, t):
    ks = np.arange(n + 1); S_vals = 2 * ks - n; pm = binom.pmf(ks, n, 0.5)
    azuma, exact_a = np.exp(-t**2 / (2 * n)), float(srw_tail(t, n))                        # (a)
    doob, exact_b = float((np.maximum(S_vals, 0) * pm).sum()) / t, float(reflection_tail(t, n))   # (b)
    print(f"n={n:5d}, t={t:3d}: Azuma {azuma:.6f} vs {exact_a:.6f} (ratio {azuma/exact_a:.2f});  "
          f"Doob {doob:.6f} vs {exact_b:.6f} (ratio {doob/exact_b:.2f})")
for n, t in [(100, 20), (400, 40), (1600, 80), (25600, 320)]:      # (c) x = t/sqrt(n) = 2 고정
    ratios(n, t)
x = 2.0
print(f"limits (x = 2): Azuma {np.exp(-x**2/2) / norm.sf(x):.2f},  Doob {1 / (2 * x * np.sqrt(2 * np.pi) * norm.sf(x)):.2f}")
# 출력: n=  100, t= 20: Azuma 0.135335 vs 0.028444 (ratio 4.76);  Doob 0.198973 vs 0.046044 (ratio 4.32)
#       n=  400, t= 40: Azuma 0.135335 vs 0.025520 (ratio 5.30);  Doob 0.199347 vs 0.045635 (ratio 4.37)
#       n= 1600, t= 80: Azuma 0.135335 vs 0.024117 (ratio 5.61);  Doob 0.199440 vs 0.045534 (ratio 4.38)
#       n=25600, t=320: Azuma 0.135335 vs 0.023089 (ratio 5.86);  Doob 0.199469 vs 0.045502 (ratio 4.38)
#       limits (x = 2): Azuma 5.95,  Doob 4.38
```

</details>

### E2. 유도 후 시뮬레이션 검증 — 둡 $L^2$ 부등식

둡 최대 부등식에서 $E[(M_n^*)^2]\le4E[M_n^2]$ ($M_n^*=\max_{k\le n}|M_k|$)을 유도하라. 그리고 $n=100$ SRW에서 $E[\max_{k\le100}|S_k|]\le2\sqrt{100}=20$과 $E[\max S_k^2]\le400$을 시뮬레이션으로 확인하라(정확값: $12.0540$, $171.2778$).

In [ ]:
# 여기에 풀이

<details><summary>정답</summary>

**유도.** $|M|$은 비음 서브마팅게일이므로 정리 1의 가운데 부등식 $\lambda P(M^*\ge\lambda)\le E[|M_n|1_{\{M^*\ge\lambda\}}]$을 쓴다. layer-cake 공식 $E[Y^2]=\int_0^\infty2\lambda P(Y\ge\lambda)\,d\lambda$에 넣으면
$$E[(M^*)^2]=\int_0^\infty2\lambda P(M^*\ge\lambda)\,d\lambda\le\int_0^\infty2E\big[|M_n|1_{\{M^*\ge\lambda\}}\big]d\lambda=2E\Big[|M_n|\int_0^{M^*}d\lambda\Big]=2E[|M_n|M^*]$$
(푸비니로 적분과 기대를 교환). 코시–슈바르츠로 $2E[|M_n|M^*]\le2\|M_n\|_2\|M^*\|_2$이므로 $\|M^*\|_2^2\le2\|M_n\|_2\|M^*\|_2$, 양변을 $\|M^*\|_2$로 나누면($<\infty$는 $M^*\le\sum_k|M_k|$로 보장) $\|M^*\|_2\le2\|M_n\|_2$, 즉 $E[(M^*)^2]\le4E[M_n^2]$. 젠센으로 $E[M^*]\le\|M^*\|_2\le2\sqrt{E[M_n^2]}=2\sqrt n$.

**검증.** 4.2의 표가 이미 답이다: $E[\max|S_k|]\approx12.05\le20$, $E[\max S_k^2]\approx171.3\le400$. 두 상한 모두 2배 안팎의 여유가 있고, 이는 SRW가 "최악의 마팅게일"과 거리가 멀다는 뜻이다.

```python
r_abs, r_sq = mc_estimate(maxAbsS), mc_estimate(maxAbsS**2)
print(f"E[max|S_k|] MC {r_abs}  exact {E_absmax_exact:.4f}  bound 20")
print(f"E[max S_k^2] MC {r_sq}  exact {E_maxsq_exact:.4f}  bound 400")
assert r_abs.mean - 4 * r_abs.se <= 2 * np.sqrt(N_STEPS) and r_sq.mean - 4 * r_sq.se <= 4 * N_STEPS
assert_close(r_abs, E_absmax_exact, k=4, name="E[max|S|]")
# 출력 예: E[max|S_k|] MC 12.0x ± 0.0x  exact 12.0540  bound 20 / E[max S_k^2] MC 17x ± x  exact 171.2778  bound 400
```

</details>

### E3. 가정을 깨보기 — 드리프트가 있는 걸음

$p=0.6$ 걸음(서브마팅게일)에 아주마 부등식을 "그냥" 적용해 $P(S_{100}\ge20)\le e^{-2}$라고 주장하면 어떻게 되는가? 시뮬레이션으로 반증하고, 올바른 **중심화** 버전을 만들어 확인하라.

In [ ]:
# 여기에 풀이

<details><summary>정답</summary>

증분의 조건부 평균이 $0.2\neq0$이므로 회프딩 보조정리의 1차 항이 사라지지 않고, 결론은 거짓이다: 정확값 $P(\mathrm{Bin}(100,0.6)\ge60)\approx0.5433\gg e^{-2}\approx0.1353$. 올바른 진술은 중심화한 마팅게일 $\tilde S_n=S_n-0.2n$에 대한 것이다. 증분 $X_k-0.2\in\{-1.2,\,0.8\}$이므로 $c=1.2$이고,
$$P(S_{100}\ge40)=P(\tilde S_{100}\ge20)\le\exp\Big(-\frac{400}{2\cdot100\cdot1.44}\Big)=e^{-25/18}\approx0.2494,$$
정확값은 $P(\mathrm{Bin}(100,0.6)\ge70)\approx0.0248$로 상한 아래에 있다. (비대칭 증분 $\{-1.2,0.8\}$에는 $c$ 대신 구간 길이 $2$를 쓰는 회프딩의 날카로운 형태 $e^{-2t^2/\sum(b_k-a_k)^2}=e^{-800/400}=e^{-2}$도 가능하다 — 여전히 성립한다.)

```python
paths06 = random_walk_paths(N_PATHS, N_STEPS, rng, p=0.6)
S06 = paths06[:, -1]
naive = mc_proportion(S06 >= 20)                  # "그냥" 적용한 주장: <= e^-2 ?
centered = mc_proportion(S06 - 0.2 * N_STEPS >= 20)   # 중심화: S_100 - 20 >= 20  <=>  S_100 >= 40
print(f"naive:    P(S_100 >= 20) MC {naive}  exact {binom.sf(59, 100, 0.6):.6f}  claimed bound e^-2 = {np.exp(-2):.4f}")
print(f"centered: P(S_100 - 20 >= 20) MC {centered}  exact {binom.sf(69, 100, 0.6):.6f}  Azuma(c=1.2) = {np.exp(-400 / 288):.4f}")
assert naive.mean - 4 * naive.se > np.exp(-2), "naive bound should be violated"
assert centered.mean + 4 * centered.se <= np.exp(-400 / 288), "centered bound should hold"
# 출력(전체 실행): naive 0.537 ± 0.004 > 0.1353 (위반); centered 0.027 ± 0.001 <= 0.2494 (성립)
```

</details>

## 7. 정리

- **둡 최대 부등식**: 비음 서브마팅게일의 최댓값 꼬리는 끝값으로 통제된다, $\lambda P(X_n^*\ge\lambda)\le E[X_n]$. 따름정리: $E[M_n^+]/\lambda$, 콜모고로프 $E[M_n^2]/\lambda^2$, $L^2$ 부등식 $\|M^*\|_2\le2\|M_n\|_2$ (SRW: $171\le400$).
- **아주마–회프딩**: 증분이 $c_k$로 유계인 마팅게일은 $P(M_n-M_0\ge t)\le\exp(-t^2/2\sum c_k^2)$ — 분포를 몰라도 가우스형 집중. 증명은 회프딩 보조정리($\cosh\le e^{x^2/2}$) + 체르노프 + 탑 성질.
- **둡 마팅게일** $E[f\mid\mathcal F_k]$는 어떤 함수든 마팅게일로 만든다; 빈 상자에서는 $E_k^{now}(1-1/n)^{m-k}$로 명시적이며 차분이 $(1-1/n)^{m-k}$로 유계다(조건부 평균 검증과 차분 상한을 숫자로 확인했다).
- **상한의 질은 $\sum c_k^2$ 대 실제 분산의 비로 결정된다**: SRW는 1(지수 최적, 상한/정확값 $\approx5$), 빈 상자는 4.5(느슨, $\approx85$배). 부등식은 근사가 아니다.
- **아주마는 마팅게일 전용**: 드리프트는 먼저 빼고, $c_k$는 결정적 상수로. SRW 최댓값의 정확값은 반사 원리가 준다(06c에서 브라운 운동판).

**trap 카드** — Q: 앞면 확률 0.6인 동전 100번의 합 $S_{100}$에 대해 아주마 부등식으로 $P(S_{100}\ge20)\le e^{-2}$라고 해도 되는가?
→ A: 안 된다. $S_n$은 마팅게일이 아니라 드리프트 0.2의 서브마팅게일이며 실제 확률은 $0.543$이다. 중심화 $S_n-0.2n$(증분 유계 $1.2$)에 적용하면 $P(S_{100}\ge40)\le0.249$가 올바른 진술이다.

**다음 노트북**: 05r — 모듈 04(재생)와 05(마팅게일)를 닫힌 책으로 복습한다. 재생-보상 비율과 선택정지 보존은 "오랜 시간 뒤 무엇이 보존되는가"라는 한 질문의 두 답이다.

`log/progress.md` 한 줄 템플릿: `- [ ] 05e 둡·아주마 부등식 — 예측 5개 중 __개 적중, 막힌 곳: __, 소요 __분`